#  Forward pass

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

__Выполнила Ларькова Дарья__

Материалы:
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann
* https://pytorch.org/docs/stable/generated/torch.matmul.html
* https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/
* https://machinelearningmastery.com/loss-and-loss-functions-for-training-deep-learning-neural-networks/
* https://kidger.site/thoughts/jaxtyping/
* https://github.com/patrick-kidger/torchtyping/tree/master

## Задачи для совместного разбора

In [2]:
import torch as th

Tensor = th.Tensor

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте нейрон с заданными весами `weights` и `bias`. Пропустите вектор `inputs` через нейрон и выведите результат.

In [3]:
class Neuron:
    def __init__(self, n_features: int):
        self.weights = th.randn(n_features)
        self.bias = th.randn(())  # скалярное смещение

    def forward(self, inputs: Tensor) -> Tensor:
        return inputs.dot(self.weights) + self.bias

In [4]:
import torch as th
inputs = th.tensor([1.0, 2.0, 3.0, 4.0])

In [5]:
neuron = Neuron(4)
neuron.forward(inputs)

tensor(-10.3641)

2\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ReLU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/f4353f4e3e484130504049599d2e7b040793e1eb)

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

In [6]:
class ReLU:
    def forward(self, inputs: Tensor) -> Tensor:
        return th.maximum(inputs, th.zeros_like(inputs))

In [7]:
matrix = th.randn(4, 3)
relu = ReLU()
result = relu.forward(matrix)
matrix, result

(tensor([[-0.5987, -0.9246, -0.3725],
         [-0.6767, -0.1342,  1.5593],
         [-0.0515,  0.3885, -0.9056],
         [-0.6228,  0.8404, -1.7970]]),
 tensor([[0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 1.5593],
         [0.0000, 0.3885, 0.0000],
         [0.0000, 0.8404, 0.0000]]))

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь MSE:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/e258221518869aa1c6561bb75b99476c4734108e)
где $Y_i$ - правильный ответ для примера $i$, $\hat{Y_i}$ - предсказание модели для примера $i$, $n$ - количество примеров в батче.

In [8]:
class MSELoss:
    def forward(self, y_pred: Tensor, y_true: Tensor) -> Tensor:
        return th.mean((y_pred - y_true) ** 2)

In [9]:
y_true = th.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = th.tensor([1.2, 1.8, 3.1, 3.9, 5.2])
mse_loss = MSELoss()
loss = mse_loss.forward(y_pred, y_true)
loss

tensor(0.0280)

## Задачи для самостоятельного решения

### Cоздание полносвязных слоев

<p class="task" id="1"></p>

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте полносвязный слой из `n_neurons` нейронов с `n_features` весами у каждого нейрона (инициализируются из стандартного нормального распределения) и опциональным вектором смещения.

$$y = xW^T + b$$

Пропустите вектор `inputs` через слой и выведите результат. Результатом прогона сквозь слой должна быть матрица размера `batch_size` x `n_neurons`.

- [ ] Проверено на семинаре

In [10]:
class Linear:
    def __init__(self, n_neurons: int, n_features: int, bias: bool = False) -> None:
        # У каждого из n_neurons нейронов есть n_features весов.
        # W имеет форму (n_neurons, n_features), как в формуле y = x W^T + b.
        self.weights = th.randn(n_neurons, n_features)
        self.bias = th.randn(n_neurons) if bias else None

    def forward(self, inputs: Tensor) -> Tensor:
        output = inputs @ self.weights.T
        if self.bias is not None:
            output = output + self.bias
        return output

In [11]:
inputs = th.randn(5, 4)

linear_no_bias = Linear(n_neurons=3, n_features=4, bias=False)
linear_with_bias = Linear(n_neurons=3, n_features=4, bias=True)

out_no_bias = linear_no_bias.forward(inputs)
out_with_bias = linear_with_bias.forward(inputs)

print("inputs shape:", inputs.shape)
print("без bias shape:", out_no_bias.shape, "bias =", linear_no_bias.bias)
print("с bias shape:", out_with_bias.shape, "bias shape =", linear_with_bias.bias.shape)
inputs

inputs shape: torch.Size([5, 4])
без bias shape: torch.Size([5, 3]) bias = None
с bias shape: torch.Size([5, 3]) bias shape = torch.Size([3])


tensor([[-1.8802,  1.2226, -0.7896,  1.0527],
        [ 1.4187,  0.0216,  2.2887, -0.2847],
        [ 0.1241, -0.5556,  1.2151, -0.7416],
        [ 0.6141, -0.4784,  0.2809, -0.8660],
        [ 0.5175, -0.6063, -0.5455,  0.9195]])

<p class="task" id="2"></p>

2\. Используя решение предыдущей задачи, создайте 2 полносвязных слоя и пропустите тензор `inputs` последовательно через эти два слоя. Количество нейронов в первом слое выберите произвольно, количество нейронов во втором слое выберите так, чтобы результатом прогона являлась матрица `batch_size x 7`.

- [ ] Проверено на семинаре

In [12]:
l1 = Linear(10, 4)
l2 = Linear(7, 10)


y = l2.forward(l1.forward(inputs))
y, y.size()

(tensor([[  1.4115,   8.8651,  -4.9001,  -7.8495,  -8.9845,  -5.7857,  -3.6506],
         [ -1.5590, -10.8704,   3.1152,   3.6077,   2.1746,   5.3562,   5.9062],
         [  1.5955,  -0.9641,  -0.6694,   9.1549,  -1.8042,  -0.7309,   3.4626],
         [ -0.8803,  -3.7328,   2.1124,   4.9353,   4.5017,   1.7502,   1.5015],
         [  1.2375,   1.9652,   0.0382,  -2.7748,  -0.1213,   1.4133,  -1.0858]]),
 torch.Size([5, 7]))

### Создание функций активации

<p class="task" id="3"></p>

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации softmax:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/6d7500d980c313da83e4117da701bf7c8f1982f5)

$$\overrightarrow{x} = (x_1, ..., x_J)$$

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации. Строки матрицы трактовать как выходы линейного слоя некоторого классификатора для 4 различных примеров. Функция должна применяться переданной на вход матрице построчно.

- [ ] Проверено на семинаре

In [13]:
class Softmax:
    def forward(self, inputs: Tensor) -> Tensor:
        shifted = inputs - th.max(inputs, dim=1, keepdim=True).values
        exp_inputs = th.exp(shifted)
        return exp_inputs / th.sum(exp_inputs, dim=1, keepdim=True)

In [14]:
matrix = th.randn(4, 3)
print("Исходная матрица:")
print(matrix)

softmax = Softmax()
result = softmax.forward(matrix)
print("\nРезультат после применения Softmax:")
print(result)
print("\nСуммы вероятностей по строкам:", result.sum(dim=1))

Исходная матрица:
tensor([[ 0.0231,  1.4055,  0.5908],
        [-0.0317, -0.0613, -2.0352],
        [ 0.4585, -0.6573, -1.0968],
        [-0.6944,  0.8925,  1.2818]])

Результат после применения Softmax:
tensor([[0.1482, 0.5904, 0.2614],
        [0.4749, 0.4611, 0.0640],
        [0.6499, 0.2129, 0.1372],
        [0.0763, 0.3731, 0.5506]])

Суммы вероятностей по строкам: tensor([1.0000, 1.0000, 1.0000, 1.0000])


<p class="task" id="4"></p>

4 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ELU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/eb23becd37c3602c4838e53f532163279192e4fd)

Создайте матрицу размера 4x3, заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

- [ ] Проверено на семинаре

In [15]:
class ELU:
    def __init__(self, alpha: float) -> None:
        self.alpha = alpha

    def forward(self, inputs: Tensor) -> Tensor:
        return th.where(inputs >= 0, inputs, self.alpha * (th.exp(inputs) - 1))

In [16]:
matrix = th.randn(4, 3)
print("Исходная матрица:")
print(matrix)
print()

elu = ELU(alpha=1.0)
result = elu.forward(matrix)
print("Результат после применения ELU (alpha=1.0):")
print(result)
print()

elu_alpha_05 = ELU(alpha=0.5)
result_alpha_05 = elu_alpha_05.forward(matrix)
print("Результат после применения ELU (alpha=0.5):")
print(result_alpha_05)
print()

Исходная матрица:
tensor([[ 0.6699,  0.7344, -1.1054],
        [ 0.0482,  0.0221,  0.4579],
        [-1.3101,  0.5357,  0.4105],
        [-0.8513,  0.5510, -0.1465]])

Результат после применения ELU (alpha=1.0):
tensor([[ 0.6699,  0.7344, -0.6689],
        [ 0.0482,  0.0221,  0.4579],
        [-0.7302,  0.5357,  0.4105],
        [-0.5731,  0.5510, -0.1363]])

Результат после применения ELU (alpha=0.5):
tensor([[ 0.6699,  0.7344, -0.3345],
        [ 0.0482,  0.0221,  0.4579],
        [-0.3651,  0.5357,  0.4105],
        [-0.2866,  0.5510, -0.0681]])



### Создание функции потерь

<p class="task" id="5"></p>

5 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь CrossEntropyLoss:

$$y_i = (y_{i,1},...,y_{i,k})$$

<img src="https://i.ibb.co/93gy1dN/Screenshot-9.png" width="200">

$$ CrossEntropyLoss = \frac{1}{n}\sum_{i=1}^{n}{L_i}$$
где $y_i$ - вектор правильных ответов для примера $i$, $\hat{y_i}$ - вектор предсказаний модели для примера $i$; $k$ - количество классов, $n$ - количество примеров в батче.

Создайте полносвязный слой с 2 нейронами и прогнать через него батч `inputs`. Полученный результат пропустите через функцию активации Softmax. Посчитайте значение функции потерь, трактуя вектор `y` как вектор правильных ответов.

- [ ] Проверено на семинаре

In [17]:
class CrossEntropyLoss:
    def forward(self, y_pred: Tensor, y_true: Tensor) -> Tensor:
        # y_pred — вероятности после Softmax.
        y_pred = th.clamp(y_pred, min=1e-8, max=1.0)
        batch_idx = th.arange(y_pred.shape[0], device=y_pred.device)
        correct_probs = y_pred[batch_idx, y_true]
        return -th.log(correct_probs).mean()

In [18]:
# Батч из 4 объектов с 3 признаками.
inputs = th.randn(4, 3)
# Для двух выходных нейронов допустимы классы только 0 и 1.
y_true = th.tensor([0, 1, 1, 0], dtype=th.long)

linear = Linear(n_neurons=2, n_features=3, bias=True)
logits = linear.forward(inputs)

softmax = Softmax()
y_pred = softmax.forward(logits)

loss_fn = CrossEntropyLoss()
loss = loss_fn.forward(y_pred, y_true)

# Проверка: ручной CrossEntropy по Softmax должен совпадать со встроенным torch.nn.functional.cross_entropy по logits.
reference_loss = th.nn.functional.cross_entropy(logits, y_true)

print("logits shape:", logits.shape)
print("probabilities shape:", y_pred.shape)
print("Суммы вероятностей:", y_pred.sum(dim=1))
print(f"Manual CrossEntropy:  {loss.item():.6f}")
print(f"PyTorch CrossEntropy: {reference_loss.item():.6f}")
print("Совпадает:", th.allclose(loss, reference_loss, atol=1e-6))

logits shape: torch.Size([4, 2])
probabilities shape: torch.Size([4, 2])
Суммы вероятностей: tensor([1.0000, 1.0000, 1.0000, 1.0000])
Manual CrossEntropy:  0.681394
PyTorch CrossEntropy: 0.681394
Совпадает: True


<p class="task" id="6"></p>

6 Модифицируйте MSE, добавив L2-регуляризацию.

$$MSE_R = MSE + \lambda\sum_{i=1}^{m}w_i^2$$

где $\lambda$ - коэффициент регуляризации; $w_i$ - веса модели.

- [ ] Проверено на семинаре

In [19]:
class MSERegularized:
    def __init__(self, lambda_: float) -> None:
        self.lambda_ = lambda_

    def data_loss(
            self,
            y_pred: Tensor,
            y_true: Tensor,
    ) -> Tensor:
        # подсчет первого слагаемого из формулы
        return th.mean((y_pred - y_true) ** 2)

    def reg_loss(self, weights: Tensor)  -> Tensor:
        # подсчет второго слагаемого из формулы
        return th.sum(weights ** 2)

    def forward(
        self,
        y_pred: Tensor,
        y_true: Tensor,
        weights: Tensor,
    ) -> Tensor:
        return self.data_loss(y_pred, y_true) + self.lambda_ * self.reg_loss(weights)

In [20]:
batch_size = 5
n_weights = 10

y_true = th.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = th.tensor([1.2, 1.8, 3.1, 3.9, 5.2])
weights = th.randn(n_weights)

for lambda_val in [0.0, 0.1, 0.5, 1.0]:
    mse_reg = MSERegularized(lambda_val)

    data_loss = mse_reg.data_loss(y_pred, y_true)
    reg_loss = mse_reg.reg_loss(weights)
    total_loss = mse_reg.forward(y_pred, y_true, weights)

    print(f"lambda = {lambda_val}:")
    print(f"  Data loss (MSE): {data_loss.item():.6f}")
    print(f"  Reg loss (L2): {reg_loss.item():.6f}")
    print(f"  Total loss: {total_loss.item():.6f}")
weights

lambda = 0.0:
  Data loss (MSE): 0.028000
  Reg loss (L2): 10.583072
  Total loss: 0.028000
lambda = 0.1:
  Data loss (MSE): 0.028000
  Reg loss (L2): 10.583072
  Total loss: 1.086307
lambda = 0.5:
  Data loss (MSE): 0.028000
  Reg loss (L2): 10.583072
  Total loss: 5.319536
lambda = 1.0:
  Data loss (MSE): 0.028000
  Reg loss (L2): 10.583072
  Total loss: 10.611072


tensor([-1.5982,  1.1447,  0.4737, -0.3503,  0.0715, -1.5650,  0.2519, -0.9502,
        -0.3857,  1.6738])

## Обратная связь
- [ ] Почта: d83073452@yandex.ru
- [ ] Gmail: 3073452@gmail.com
- [ ] Telegram: @sleepingzzzZ